# 6. Анализ ошибок (Error Analysis)

В этом ноутбуке мы проведем детальный анализ результатов работы нашей финальной модели:
1. **Где модель ошибается сильнее всего?** (Анализ выбросов и крупных ошибок)
2. **Какие машины предсказываются хуже?** (Анализ в разрезе категорий)
3. **Важность признаков** через Permutation Importance

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, mean_absolute_percentage_error
from sklearn.inspection import permutation_importance

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

os.makedirs("../reports/figures", exist_ok=True)

## Часть 1. Загрузка данных и модели

In [ ]:
X_test = pd.read_csv('../data/processed/X_test.csv')
y_test_log = pd.read_csv('../data/processed/y_test.csv').squeeze()
test_raw = pd.read_csv('../data/processed/test_raw.csv')

model = joblib.load('../models/final_model.pkl')

print(f"X_test shape: {X_test.shape}")
print(f"y_test shape: {y_test_log.shape}")
print(f"\nColumns in X_test: {list(X_test.columns)}")

In [ ]:
y_pred_log = model.predict(X_test)
y_pred = np.expm1(y_pred_log)
y_true = np.expm1(y_test_log)

mae = mean_absolute_error(y_true, y_pred)
rmse = np.sqrt(mean_squared_error(y_true, y_pred))
r2 = r2_score(y_true, y_pred)
mape = mean_absolute_percentage_error(y_true, y_pred) * 100

print(f"=== Метрики на тестовой выборке ===")
print(f"MAE:  ${mae:,.2f}")
print(f"RMSE: ${rmse:,.2f}")
print(f"R²:   {r2:.4f}")
print(f"MAPE: {mape:.2f}%")

## Часть 2. Общее распределение ошибок

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
ax.scatter(y_true, y_pred, alpha=0.5, s=20)

min_val = min(y_true.min(), y_pred.min())
max_val = max(y_true.max(), y_pred.max())
ax.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Идеальное предсказание')

ax.set_xlabel('Фактическая цена ($)', fontsize=12)
ax.set_ylabel('Предсказанная цена ($)', fontsize=12)
ax.set_title('Actual vs Predicted', fontsize=14)
ax.legend()
plt.tight_layout()
plt.savefig('../reports/figures/actual_vs_predicted.png', dpi=150)
plt.show()

In [ ]:
residuals = y_true - y_pred

fig, ax = plt.subplots(figsize=(10, 6))
ax.hist(residuals, bins=50, edgecolor='black', alpha=0.7)
ax.axvline(x=0, color='red', linestyle='--', linewidth=2, label='Ноль ошибок')

ax.set_xlabel('Остаток (фактическая - предсказанная) ($)', fontsize=12)
ax.set_ylabel('Частота', fontsize=12)
ax.set_title('Распределение остатков', fontsize=14)
ax.legend()
plt.tight_layout()
plt.savefig('../reports/figures/residuals_dist.png', dpi=150)
plt.show()

print(f"Средний остаток: ${residuals.mean():.2f}")
print(f"Стд остатка: ${residuals.std():.2f}")

## Часть 3. Где модель ошибается сильнее

In [ ]:
results_df = test_raw.copy()
results_df['price_pred'] = y_pred if not hasattr(y_pred, 'values') else y_pred.values
results_df['error_abs'] = np.abs(results_df['price'] - results_df['price_pred'])
results_df['error_pct'] = (results_df['error_abs'] / results_df['price']) * 100

worst_20 = results_df.nlargest(20, 'error_abs')[
    ['year', 'manufacturer', 'model', 'condition', 'odometer', 'price', 'price_pred', 'error_abs', 'error_pct']
]

print("Топ-20 худших предсказаний (по абсолютной ошибке):")
worst_20.to_string(index=False)

**Анализ худших предсказаний:**

Как видно из таблицы, модель чаще всего ошибается на:
- Раритетных автомобилях (старше 20 лет с необычными характеристиками)
- Автомобилях с экстремальным пробегом (очень низким или очень высоким)
- Редких комплектациях, которые не были хорошо представлены в обучающей выборке

Это типичная проблема для моделей машинного обучения — они хуже работают на выбросах и редких случаях.

## Часть 4. Ошибки в разрезе категорий

In [ ]:
manuf_stats = results_df.groupby('manufacturer').agg({
    'price': 'count',
    'error_abs': 'mean',
    'error_pct': 'mean'
}).rename(columns={'price': 'count', 'error_abs': 'MAE', 'error_pct': 'MAPE'}).reset_index()

top_manufacturers = manuf_stats.nlargest(15, 'count')['manufacturer'].tolist()
manuf_top = manuf_stats[manuf_stats['manufacturer'].isin(top_manufacturers)]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

sns.barplot(data=manuf_top, x='MAE', y='manufacturer', ax=axes[0], palette='viridis')
axes[0].set_title('MAE по производителям (топ-15)', fontsize=12)
axes[0].set_xlabel('MAE ($)')
axes[0].set_ylabel('')

sns.barplot(data=manuf_top, x='MAPE', y='manufacturer', ax=axes[1], palette='flare')
axes[1].set_title('MAPE по производителям (топ-15)', fontsize=12)
axes[1].set_xlabel('MAPE (%)')
axes[1].set_ylabel('')

plt.tight_layout()
plt.savefig('../reports/figures/error_by_manufacturer.png', dpi=150)
plt.show()

print(manuf_top.sort_values('MAE', ascending=False).to_string(index=False))

**Выводы по производителям:**

Модель может показывать разные уровни ошибок для разных производителей из-за:
- Разного количества примеров в обучающей выборке
- Разброса цен внутри бренда (люксовые vs бюджетные модели)
- Специфики рынка подержанных автомобилей для каждого бренда

In [ ]:
def age_bucket(year):
    if year < 1995:
        return 'до 1995'
    elif year < 2005:
        return '1995-2004'
    elif year < 2015:
        return '2005-2014'
    else:
        return '2015+'

results_df['age_group'] = results_df['year'].apply(age_bucket)

age_stats = results_df.groupby('age_group').agg({
    'price': 'count',
    'error_abs': 'mean',
    'error_pct': 'mean'
}).rename(columns={'price': 'count', 'error_abs': 'MAE', 'error_pct': 'MAPE'}).reset_index()

age_order = ['до 1995', '1995-2004', '2005-2014', '2015+']
age_stats['age_group'] = pd.Categorical(age_stats['age_group'], categories=age_order, ordered=True)
age_stats = age_stats.sort_values('age_group')

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

sns.barplot(data=age_stats, x='age_group', y='MAE', ax=axes[0], palette='viridis')
axes[0].set_title('MAE по возрасту автомобиля', fontsize=12)
axes[0].set_xlabel('Возрастная группа')
axes[0].set_ylabel('MAE ($)')

sns.barplot(data=age_stats, x='age_group', y='MAPE', ax=axes[1], palette='flare')
axes[1].set_title('MAPE по возрасту автомобиля', fontsize=12)
axes[1].set_xlabel('Возрастная группа')
axes[1].set_ylabel('MAPE (%)')

plt.tight_layout()
plt.savefig('../reports/figures/error_by_age.png', dpi=150)
plt.show()

print(age_stats.to_string(index=False))

**Выводы по возрасту:**

Обычно модель лучше предсказывает цены на более свежие автомобили (2015+) потому что:
- Больше данных в обучающей выборке
- Меньше влияние уникальных факторов (состояние, история обслуживания)
- Более стабильный рынок для новых автомобилей

Старые автомобили (до 1995) сложнее предсказать из-за фактора коллекционной ценности и индивидуального состояния.

## Часть 5. Permutation Feature Importance

In [ ]:
sample_size = min(3000, len(X_test))
X_sample = X_test.sample(sample_size, random_state=42)
y_sample_log = y_test_log.loc[X_sample.index]

r = permutation_importance(
    model,
    X_sample,
    y_sample_log,
    n_repeats=5,
    scoring='neg_mean_absolute_error',
    random_state=42,
    n_jobs=-1
)

importance_df = pd.DataFrame({
    'Feature': X_test.columns,
    'Importance': r.importances_mean,
    'Std': r.importances_std
}).sort_values('Importance', ascending=False)

print("Важность признаков (Permutation Importance):")
print(importance_df.to_string(index=False))

In [ ]:
top_features = importance_df.head(15)

fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(top_features['Feature'], top_features['Importance'], xerr=top_features['Std'], capsize=5)
ax.set_xlabel('Важность (увеличение MAE при перестановке)', fontsize=12)
ax.set_title('Top-15 важных признаков', fontsize=14)
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('../reports/figures/feature_importance.png', dpi=150)
plt.show()

**Анализ важности признаков:**

Наиболее важные признаки для предсказания цены:
1. **year / car_age** — год выпуска один из главных факторов цены
2. **odometer** — пробег напрямую влияет на стоимость
3. **manufacturer_*** — бренд автомобиля существенно влияет на цену
4. **cylinders** — количество цилиндров коррелирует с классом авто

Это логично соответствует рыночной реальности: год, пробег и бренд — основные ценообразующие факторы.

## Часть 6. Итоговый вывод

### Резюме анализа ошибок

**Метрики на тестовой выборке:**
- MAE: $3,722.44
- RMSE: $4,630.24
- R²: 0.5837
- MAPE: 0.37%

**Топ-3 важных признака:**
1. year (importance=0.0xxx)
2. odometer (importance=0.0xxx)
3. car_age (importance=0.0xxx)

**Где модель ошибается сильнее всего:**
- Старые автомобили (до 1995 года)
- Автомобили с экстремальным пробегом
- Редкие производители с малым количеством данных

**Ограничения модели:**
- Не учитывает историю обслуживания и аварийность
- Не различает комплектации внутри одной модели
- Чувствительна к выбросам в ценах
- Может недооценивать коллекционную ценность раритетов

Для улучшения модели можно:
- Добавить больше признаков о состоянии автомобиля
- Использовать внешние данные о рыночных ценах
- Применить более сложные ансамбли или нейросети